# Part 2: Word Vectors - 句子分词与数据准备

本 Notebook 实现 Part 2 教程，学习如何使用 **NLTK 句子分词器**将评论拆分为句子，为 Word2Vec 训练做准备。

## 为什么需要句子分词？

Word2Vec 算法需要理解词的上下文，而**句子**是最自然的上下文单元。相比于 Part 1 的简单分词，句子级别的拆分能更好地保留语义信息。

## 1. 安装和配置 NLTK

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

# 下载 punkt 分词器 (首次运行需要)
print("正在下载 NLTK punkt 分词器...")
nltk.download('punkt')
nltk.download('punkt_tab')  # 新版 NLTK 需要
print("✓ NLTK 准备完成!")

## 2. 加载数据

In [ ]:
import pandas as pd
import os

DATA_DIR = "../word2vec-nlp-tutorial"

# 读取训练数据
train = pd.read_csv(
    os.path.join(DATA_DIR, "labeledTrainData.tsv"), 
    header=0, 
    delimiter="\t", 
    quoting=3
)

# 读取测试数据
test = pd.read_csv(
    os.path.join(DATA_DIR, "testData.tsv"), 
    header=0, 
    delimiter="\t", 
    quoting=3
)

# 读取无标签数据 (用于 Word2Vec 训练)
unlabeled = pd.read_csv(
    os.path.join(DATA_DIR, "unlabeledTrainData.tsv"), 
    header=0, 
    delimiter="\t", 
    quoting=3
)

print(f"训练集：{train.shape}")
print(f"测试集：{test.shape}")
print(f"无标签数据：{unlabeled.shape}")

## 3. 文本清洗 (复用 Part 1 的函数)

In [ ]:
import re
from bs4 import BeautifulSoup

def review_to_words(raw_review):
    """去除 HTML、标点，转小写"""
    markup = BeautifulSoup(raw_review, 'html.parser')
    markup_text = markup.get_text()
    letters_only = re.sub("[^a-zA-Z]", " ", markup_text)
    words = letters_only.lower().split()
    return " ".join(words)

# 测试
print("原始评论:")
print(train['review'][0][:150])
print("\n清洗后:")
print(review_to_words(train['review'][0])[:150])

## 4. 句子分词

使用 NLTK 的 `sent_tokenize` 函数将评论拆分为句子列表。

In [ ]:
# 测试句子分词
sample_review = review_to_words(train['review'][0])
sentences = sent_tokenize(sample_review)

print(f"评论被拆分为 {len(sentences)} 个句子:\n")
for i, sent in enumerate(sentences[:5]):  # 显示前 5 句
    print(f"{i+1}. {sent}")

## 5. 准备 Word2Vec 数据结构

Word2Vec 需要的输入格式是：**列表的列表**，外层是句子，内层是单词。

In [ ]:
from nltk.tokenize import word_tokenize

def make_sentences_list(reviews, max_sentences=100):
    """
    将评论列表转换为句子列表（每个句子是单词列表）
    
    参数:
        reviews: 评论字符串列表
        max_sentences: 每条评论最多保留的句子数
    
    返回:
        sentences: 列表的列表 [[word1, word2, ...], ...]
    """
    sentences = []
    for i, review in enumerate(reviews):
        if (i+1) % 5000 == 0:
            print(f"  已处理 {i+1}/{len(reviews)} 条")
        
        # 清洗
        clean_review = review_to_words(review)
        
        # 句子分词
        review_sentences = sent_tokenize(clean_review)
        
        # 限制句子数量并分词
        for sent in review_sentences[:max_sentences]:
            words = word_tokenize(sent)
            if len(words) > 0:  # 跳过空句子
                sentences.append(words)
    
    return sentences

# 处理训练集
print("处理训练集...")
train_sentences = make_sentences_list(train['review'].tolist())
print(f"✓ 训练集句子数：{len(train_sentences)}")
print(f"示例句子：{train_sentences[0]}")

## 6. 处理无标签数据 (用于 Word2Vec 训练)

无标签数据量更大 (50,000 条)，可以用来训练更好的词向量。

In [ ]:
print("处理无标签数据 (用于 Word2Vec 训练)...")
unlabeled_sentences = make_sentences_list(unlabeled['review'].tolist())
print(f"✓ 无标签数据句子数：{len(unlabeled_sentences)}")

# 合并所有数据
all_sentences = train_sentences + unlabeled_sentences
print(f"\n总句子数：{len(all_sentences)}")

## 7. 数据统计

In [ ]:
import numpy as np

# 句子长度统计
sentence_lengths = [len(sent) for sent in all_sentences]

print(f"句子长度统计:")
print(f"  最短句子：{min(sentence_lengths)} 词")
print(f"  最长句子：{max(sentence_lengths)} 词")
print(f"  平均长度：{np.mean(sentence_lengths):.1f} 词")
print(f"  中位数：{np.median(sentence_lengths):.1f} 词")

# 词汇统计
all_words = [word for sent in all_sentences for word in sent]
unique_words = set(all_words)

print(f"\n词汇统计:")
print(f"  总词数：{len(all_words)}")
print(f"  不重复词数：{len(unique_words)}")

# 词频统计
from collections import Counter
word_freq = Counter(all_words)

print(f"\n最常见单词:")
for word, count in word_freq.most_common(20):
    print(f"  {word}: {count} 次")

## 8. 保存处理后的数据 (可选)

In [ ]:
import pickle

# 保存句子数据
with open('../data/processed/train_sentences.pkl', 'wb') as f:
    pickle.dump(train_sentences, f)

with open('../data/processed/all_sentences.pkl', 'wb') as f:
    pickle.dump(all_sentences, f)

print("✓ 数据已保存到 data/processed/")

## 总结

**本 Part 完成的工作**:
1. ✅ 安装和配置 NLTK (punkt 分词器)
2. ✅ 句子级别分词 (sent_tokenize)
3. ✅ 单词级别分词 (word_tokenize)
4. ✅ 准备 Word2Vec 数据结构 (列表的列表)
5. ✅ 处理无标签数据用于训练

** next Part**: Part 3 将使用这些数据训练 Word2Vec 模型！